In [1]:
import os
import opf

import numpy as np
import pandas as pd
import pyomo.environ as pyo

from matpower import path_matpower_cases, start_instance
from matpowercaseframes import CaseFrames


In [2]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 300)
pd.set_option('display.precision', 2)
pd.set_option('display.float_format', '{:.2f}'.format)

## Original case9.m

In [3]:
case_path = os.path.join(path_matpower_cases, 'case9.m')

In [4]:
model = opf.build_model('acopf')
network = opf.parse_file(case_path)
model.instantiate(network)
result = model.solve(
    solver_option={'print_level' : 5},
    tee=True
)

build model... end
instantiate model... end
Ipopt 3.14.16: print_level=5


******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit https://github.com/coin-or/Ipopt
******************************************************************************

This is Ipopt version 3.14.16, running with linear solver MUMPS 5.6.2.

Number of nonzeros in equality constraint Jacobian...:      223
Number of nonzeros in inequality constraint Jacobian.:       54
Number of nonzeros in Lagrangian Hessian.............:      102

Total number of variables............................:       60
                     variables with only lower bounds:        0
                variables with lower and upper bounds:       15
                     variables with only upper bounds:        0
Total number of eq

In [5]:
print(
    f"Status: {result['termination_status']}\n"
    f"Objective function: {result['obj_cost']}\n"
    f"Output power: {result['sol']['primal']['pg']}\n"
    f"Computation time: {result['time']}"
)

Status: optimal
Objective function: 5296.686202481215
Output power: {'1': 0.8979870766526404, '2': 1.3432060074581076, '3': 0.9418738042409803}
Computation time: 0.04003190994262695


In [6]:
cf = CaseFrames(case_path)
cf.infer_numpy()

In [7]:
def short_int_df_index(df):
    df.index = [int(i) for i in df.index]
    return df.sort_index()

In [8]:
df_pyopf_branch = short_int_df_index(pd.DataFrame({
    'pf_from': result['sol']['primal']['pf_from'],
    'qf_from': result['sol']['primal']['qf_from'],
    'pf_to': result['sol']['primal']['pf_to'],
    'qf_to': result['sol']['primal']['qf_to'],
}))
df_pyopf_gen = short_int_df_index(pd.DataFrame({
    'pg': result['sol']['primal']['pg'],
    'qg': result['sol']['primal']['qg'],
}))
df_pyopf_bus = short_int_df_index(pd.DataFrame({
    'vm': result['sol']['primal']['vm'],
    'va': result['sol']['primal']['va'],
}))
df_pyopf_branch[['pf_from', 'qf_from', 'pf_to', 'qf_to']] = df_pyopf_branch[['pf_from', 'qf_from', 'pf_to', 'qf_to']] * cf.baseMVA
df_pyopf_gen[['pg', 'qg']] = df_pyopf_gen[['pg', 'qg']] * cf.baseMVA
df_pyopf_bus['va'] = df_pyopf_bus['va'] * 180 / np.pi

# print(df_pyopf_branch)
# print(df_pyopf_gen)
# print(df_pyopf_bus)

cf.branch[['PF', 'QF', 'PT', 'QT']] = df_pyopf_branch[['pf_from', 'qf_from', 'pf_to', 'qf_to']]
cf.bus[['VM', 'VA']] = df_pyopf_bus[['vm', 'va']]
cf.gen[['PG', 'QG']] = df_pyopf_gen[['pg', 'qg']]

In [9]:
print(cf.branch)
print(cf.bus)
print(cf.gen)

   F_BUS  T_BUS  BR_R  BR_X  BR_B  RATE_A  RATE_B  RATE_C  TAP  SHIFT  BR_STATUS  ANGMIN  ANGMAX      PF     QF     PT     QT
1      1      4  0.00  0.06  0.00     250     250     250    0      0          1    -360     360   89.80  12.97 -89.80  -9.05
2      4      5  0.02  0.09  0.16     250     250     250    0      0          1    -360     360   35.22  -3.89 -35.04 -13.88
3      5      6  0.04  0.17  0.36     150     150     150    0      0          1    -360     360  -54.96 -16.12  55.97 -22.19
4      3      6  0.00  0.06  0.00     300     300     300    0      0          1    -360     360   94.19 -22.63 -94.19  27.29
5      6      7  0.01  0.10  0.21     150     150     150    0      0          1    -360     360   38.22  -5.10 -38.07 -18.68
6      7      8  0.01  0.07  0.15     250     250     250    0      0          1    -360     360  -61.93 -16.32  62.21   0.82
7      8      2  0.00  0.06  0.00     250     250     250    0      0          1    -360     360 -134.32   9.33 134.32

In [10]:
m = start_instance()

In [11]:
sol = m.runpf(cf.to_dict(), verbose=False)

In [12]:
m.exit()

In [13]:
cf_runpf = CaseFrames(sol)
print(cf_runpf.branch)
print(cf_runpf.bus)
print(cf_runpf.gen)

   F_BUS  T_BUS  BR_R  BR_X  BR_B  RATE_A  RATE_B  RATE_C  TAP  SHIFT  BR_STATUS  ANGMIN  ANGMAX      PF     QF     PT     QT
1   1.00   4.00  0.00  0.06  0.00  250.00  250.00  250.00 0.00   0.00       1.00 -360.00  360.00   90.31  25.08 -90.31 -20.40
2   4.00   5.00  0.02  0.09  0.16  250.00  250.00  250.00 0.00   0.00       1.00 -360.00  360.00   35.30   0.23 -35.09 -15.54
3   5.00   6.00  0.04  0.17  0.36  150.00  150.00  150.00 0.00   0.00       1.00 -360.00  360.00  -54.91 -14.46  56.06 -18.05
4   3.00   6.00  0.00  0.06  0.00  300.00  300.00  300.00 0.00   0.00       1.00 -360.00  360.00   94.19 -11.52 -94.19  16.54
5   6.00   7.00  0.01  0.10  0.21  150.00  150.00  150.00 0.00   0.00       1.00 -360.00  360.00   38.13   1.51 -37.95 -21.94
6   7.00   8.00  0.01  0.07  0.15  250.00  250.00  250.00 0.00   0.00       1.00 -360.00  360.00  -62.05 -13.06  62.37   0.21
7   8.00   2.00  0.00  0.06  0.00  250.00  250.00  250.00 0.00   0.00       1.00 -360.00  360.00 -134.32   8.29 134.32

## Check Splitting Branch

In [14]:
N = 2
index = cf.branch.index.tolist()
for i in range(1, N):
    index += list(cf.branch.index + cf.branch.index[-1] * i)
cf.branch = pd.concat([cf.branch] * N, ignore_index=True)
cf.branch.loc[:, ["BR_R", "BR_X"]] = cf.branch.loc[:, ["BR_R", "BR_X"]] * N
cf.branch.loc[:, ["BR_B", "RATE_A", "RATE_B", "RATE_C"]] = (
    cf.branch.loc[:, ["BR_B", "RATE_A", "RATE_B", "RATE_C"]] / N
)
cf.branch.index = index
cf.branch

,F_BUS,T_BUS,BR_R,BR_X,BR_B,RATE_A,RATE_B,RATE_C,TAP,SHIFT,BR_STATUS,ANGMIN,ANGMAX,PF,QF,PT,QT
1,1,4,0.00,0.12,0.00,125,125,125,0,0,1,-360,360,89.80,12.97,-89.80,-9.05
2,4,5,0.03,0.18,0.08,125,125,125,0,0,1,-360,360,35.22,-3.89,-35.04,-13.88
3,5,6,0.08,0.34,0.18,75,75,75,0,0,1,-360,360,-54.96,-16.12,55.97,-22.19
4,3,6,0.00,0.12,0.00,150,150,150,0,0,1,-360,360,94.19,-22.63,-94.19,27.29
5,6,7,0.02,0.20,0.10,75,75,75,0,0,1,-360,360,38.22,-5.10,-38.07,-18.68
6,7,8,0.02,0.14,0.07,125,125,125,0,0,1,-360,360,-61.93,-16.32,62.21,0.82
7,8,2,0.00,0.12,0.00,125,125,125,0,0,1,-360,360,-134.32,9.33,134.32,0.03
8,8,9,0.06,0.32,0.15,125,125,125,0,0,1,-360,360,72.11,-10.15,-70.72,-18.92
9,9,4,0.02,0.17,0.09,125,125,125,0,0,1,-360,360,-54.28,-31.08,54.58,12.94
10,1,4,0.00,0.12,0.00,125,125,125,0,0,1,-360,360,89.80,12.97,-89.80,-9.05


In [15]:
from opf.io.common import make_per_unit
from opf.io.matpower import MP_BUS_COLUMNS, MP_GEN_COLUMNS, MP_BRANCH_COLUMNS, mp2data

MP_COLUMNS = {
    'bus': MP_BUS_COLUMNS,
    'gen': MP_GEN_COLUMNS,
    'branch': MP_BRANCH_COLUMNS,
}
def mpc2pyopf(mpc, name=''):
    mp_data = {
        'source_type': 'matpower',
        'name': name,
    }
    for attribute in mpc:
        # TODO: instead of enumerate, it should based on original matpower indexing
        if attribute in ('bus', 'branch', 'gen'):
            mp_data[attribute] = [
                {MP_COLUMNS[attribute][i][0]: MP_COLUMNS[attribute][i][1](val)
                 for i, val in enumerate(row)}
                 for row in mpc[attribute]
            ]
            if attribute == 'bus':
                for row in mp_data[attribute]:
                    row['id'] = int(row['bus_i'])
            elif attribute == 'gen':
                for id, row in enumerate(mp_data[attribute], 1):
                    row['gen_bus'] = str(int(float(row['gen_bus'])))
                    row['id'] = id
            else:  # branch
                for id, row in enumerate(mp_data[attribute], 1):
                    row['f_bus'] = str(int(float(row['f_bus'])))
                    row['t_bus'] = str(int(float(row['t_bus'])))
                    row['id'] = id
        elif attribute == 'gencost':
            mp_data[attribute] = []
            for id, row in enumerate(mpc[attribute], 1):
                model = int(row[0])
                if model != 2:
                    msg = (f"Generator cost model {model} is not supported. It should"
                            " be model=2.")
                    raise ValueError(msg)
                startup = float(row[1])
                shutdown = float(row[2])
                ncost = int(row[3])
                costs = [float(row[4+c]) for c in range(ncost)]
                entry = {
                    'id': int(id),
                    'model': model,
                    'startup': startup,
                    'shutdown': shutdown,
                    'cost': costs
                }
                mp_data[attribute].append(entry)
        else:
            mp_data[attribute] = mpc[attribute]
    return mp_data

In [16]:
mp_data = mpc2pyopf(cf.to_dict())
network = mp2data(mp_data)
make_per_unit(network)
network['preprocessed'] = False

In [17]:
model.instantiate(network)
result = model.solve(
    solver_option={'print_level' : 5},
    tee=True
)

instantiate model... end
Ipopt 3.14.16: print_level=5


******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit https://github.com/coin-or/Ipopt
******************************************************************************

This is Ipopt version 3.14.16, running with linear solver MUMPS 5.6.2.

Number of nonzeros in equality constraint Jacobian...:      439
Number of nonzeros in inequality constraint Jacobian.:      108
Number of nonzeros in Lagrangian Hessian.............:      138

Total number of variables............................:       96
                     variables with only lower bounds:        0
                variables with lower and upper bounds:       15
                     variables with only upper bounds:        0
Total number of equality constraints.

/Users/macbookair/Documents/Git/PyOPF/opf/core/base.py:172: RuntimeWarning: instance is already created. instantiating again will destroy the previous instance
  warnings.warn("instance is already created. instantiating again will destroy the previous instance", RuntimeWarning)


In [18]:
print(
    f"Status: {result['termination_status']}\n"
    f"Objective function: {result['obj_cost']}\n"
    f"Output power: {result['sol']['primal']['pg']}\n"
    f"Computation time: {result['time']}"
)

Status: optimal
Objective function: 5296.686202429669
Output power: {'1': 0.8979870768644013, '2': 1.3432060073358951, '3': 0.9418738041238321}
Computation time: 0.016273021697998047


In [19]:
df_pyopf_branch = short_int_df_index(pd.DataFrame({
    'pf_from': result['sol']['primal']['pf_from'],
    'qf_from': result['sol']['primal']['qf_from'],
    'pf_to': result['sol']['primal']['pf_to'],
    'qf_to': result['sol']['primal']['qf_to'],
}))
df_pyopf_gen = short_int_df_index(pd.DataFrame({
    'pg': result['sol']['primal']['pg'],
    'qg': result['sol']['primal']['qg'],
}))
df_pyopf_bus = short_int_df_index(pd.DataFrame({
    'vm': result['sol']['primal']['vm'],
    'va': result['sol']['primal']['va'],
}))
df_pyopf_branch[['pf_from', 'qf_from', 'pf_to', 'qf_to']] = df_pyopf_branch[['pf_from', 'qf_from', 'pf_to', 'qf_to']] * cf.baseMVA
df_pyopf_gen[['pg', 'qg']] = df_pyopf_gen[['pg', 'qg']] * cf.baseMVA
df_pyopf_bus['va'] = df_pyopf_bus['va'] * 180 / np.pi

# print(df_pyopf_branch)
# print(df_pyopf_gen)
# print(df_pyopf_bus)

cf.branch[['PF', 'QF', 'PT', 'QT']] = df_pyopf_branch[['pf_from', 'qf_from', 'pf_to', 'qf_to']]
cf.bus[['VM', 'VA']] = df_pyopf_bus[['vm', 'va']]
cf.gen[['PG', 'QG']] = df_pyopf_gen[['pg', 'qg']]

In [20]:
print(cf.branch)
print(cf.bus)
print(cf.gen)

    F_BUS  T_BUS  BR_R  BR_X  BR_B  RATE_A  RATE_B  RATE_C  TAP  SHIFT  BR_STATUS  ANGMIN  ANGMAX     PF     QF     PT     QT
1       1      4  0.00  0.12  0.00     125     125     125    0      0          1    -360     360  44.90   6.48 -44.90  -4.52
2       4      5  0.03  0.18  0.08     125     125     125    0      0          1    -360     360  17.61  -1.95 -17.52  -6.94
3       5      6  0.08  0.34  0.18      75      75      75    0      0          1    -360     360 -27.48  -8.06  27.98 -11.10
4       3      6  0.00  0.12  0.00     150     150     150    0      0          1    -360     360  47.09 -11.32 -47.09  13.65
5       6      7  0.02  0.20  0.10      75      75      75    0      0          1    -360     360  19.11  -2.55 -19.03  -9.34
6       7      8  0.02  0.14  0.07     125     125     125    0      0          1    -360     360 -30.97  -8.16  31.10   0.41
7       8      2  0.00  0.12  0.00     125     125     125    0      0          1    -360     360 -67.16   4.67  67.16

In [21]:
m = start_instance()

In [22]:
sol = m.runpf(cf.to_dict(), verbose=False)

In [23]:
m.exit()

In [24]:
cf_runpf = CaseFrames(sol)
print(cf_runpf.branch)
print(cf_runpf.bus)
print(cf_runpf.gen)

    F_BUS  T_BUS  BR_R  BR_X  BR_B  RATE_A  RATE_B  RATE_C  TAP  SHIFT  BR_STATUS  ANGMIN  ANGMAX     PF     QF     PT     QT
1    1.00   4.00  0.00  0.12  0.00  125.00  125.00  125.00 0.00   0.00       1.00 -360.00  360.00  45.16  12.54 -45.16 -10.20
2    4.00   5.00  0.03  0.18  0.08  125.00  125.00  125.00 0.00   0.00       1.00 -360.00  360.00  17.65   0.11 -17.55  -7.77
3    5.00   6.00  0.08  0.34  0.18   75.00   75.00   75.00 0.00   0.00       1.00 -360.00  360.00 -27.45  -7.23  28.03  -9.02
4    3.00   6.00  0.00  0.12  0.00  150.00  150.00  150.00 0.00   0.00       1.00 -360.00  360.00  47.09  -5.76 -47.09   8.27
5    6.00   7.00  0.02  0.20  0.10   75.00   75.00   75.00 0.00   0.00       1.00 -360.00  360.00  19.07   0.75 -18.98 -10.97
6    7.00   8.00  0.02  0.14  0.07  125.00  125.00  125.00 0.00   0.00       1.00 -360.00  360.00 -31.02  -6.53  31.18   0.10
7    8.00   2.00  0.00  0.12  0.00  125.00  125.00  125.00 0.00   0.00       1.00 -360.00  360.00 -67.16   4.14  67.16

Conclusion: Formula used by PyOPF support parallel lines.